# Gradient Boosting and Modern Boosting Libraries

> Understand boosting with a scikit-learn baseline and optional specialist libraries.

**Track:** Machine Learning Roadmap · **Language:** English

## Learning goals
- Understand boosting as a sequence of learners that correct prior errors.
- Fit a native scikit-learn histogram gradient boosting baseline.
- Know when XGBoost, LightGBM, or CatBoost may be useful and how to compare them fairly.

Boosted trees are strong candidates for many tabular datasets, but there is no universally best model. Compare baselines using the same split, metric, tuning budget, and preprocessing assumptions.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score

X, y = make_classification(n_samples=1200, n_features=18, n_informative=7, n_redundant=4, weights=[.78,.22], random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, stratify=y, random_state=42)
models = {
    "Random forest": RandomForestClassifier(n_estimators=180, min_samples_leaf=3, random_state=42, n_jobs=1),
    "Histogram gradient boosting": HistGradientBoostingClassifier(max_iter=120, learning_rate=.08, max_leaf_nodes=15, random_state=42),
}
results=[]
for name, estimator in models.items():
    estimator.fit(X_train, y_train)
    results.append({"model": name, "ROC-AUC": roc_auc_score(y_test, estimator.predict_proba(X_test)[:,1])})
display(pd.DataFrame(results).sort_values("ROC-AUC", ascending=False).round(3))

,model,ROC-AUC
0,Random forest,0.928
1,Histogram gradient boosting,0.918


## Optional specialist libraries
Install the model family you want to evaluate; avoid installing every large dependency by default.

```bash
python -m pip install xgboost
python -m pip install lightgbm
python -m pip install catboost
```

- **XGBoost:** regularized gradient boosting with a broad ecosystem and CPU/GPU support.
- **LightGBM:** gradient boosting designed for efficient training, often attractive for larger tabular datasets.
- **CatBoost:** designed to handle categorical features conveniently with its native workflow.

Official references: [XGBoost Python docs](https://xgboost.readthedocs.io/en/stable/python/), [LightGBM docs](https://lightgbm.readthedocs.io/), [CatBoost docs](https://catboost.ai/en/docs/). API details evolve; pin the versions used in the project.

In [2]:
# Optional model availability check; does not install packages or download data.
import importlib.util
for package in ["xgboost", "lightgbm", "catboost"]:
    print(f"{package:10} installed:", importlib.util.find_spec(package) is not None)

xgboost    installed: True
lightgbm   installed: True
catboost   installed: True


## Tuning principles
- Tune a small number of meaningful parameters before launching a huge search.
- `learning_rate` and number of boosting iterations interact.
- Evaluate the whole selection process on validation/CV data; keep the test set untouched.
- Compare training speed, inference latency, model size, calibration, interpretability, and operational reliability—not only AUC.

## Practice
1. Add `max_leaf_nodes` alternatives to the scikit-learn model.
2. Design a fair comparison table for three boosting libraries.
3. State why missing-value and categorical handling differs by implementation.